[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/04-choropleth-maps.ipynb)

# Choropleth Maps

`create_map` generates choropleth maps where geographic areas are colored by a data variable. It supports PNG/PDF/SVG image output, GeoJSON/Shapefile export, and interactive HTML maps (with optional `folium` dependency).

In this notebook you will learn how to:

1. Build the data pipeline (isochrone → blocks → census → merge)
2. Create a basic population map
3. Map different demographic columns
4. Use custom colormaps
5. Change basemap styles
6. Overlay an isochrone boundary
7. Overlay point markers
8. Show statistics on the map
9. Export to GeoJSON
10. Save to a file
11. Create an interactive HTML map

## Setup

In [ ]:
# Uncomment to install on Google Colab:
# !pip install 'socialmapper[interactive] @ git+https://github.com/mihiarc/socialmapper.git'

from socialmapper import create_isochrone, get_census_blocks, get_census_data, create_map
from IPython.display import Image, display, HTML

## 1. Build the Data Pipeline

Every choropleth map needs geographic data with at least one numeric column. Here we build the standard pipeline.

In [ ]:
# Step 1: Create isochrone
iso = create_isochrone("Portland, OR", travel_time=15, travel_mode="drive")
print(f"Isochrone area: {iso['properties']['area_sq_km']:.1f} sq km")

# Step 2: Get block groups
blocks = get_census_blocks(polygon=iso)
print(f"Block groups: {len(blocks)}")

# Step 3: Get census data
variables = ["population", "median_income", "median_age", "housing_units"]
census = get_census_data(iso, variables=variables)

# Step 4: Merge — attach census values to each block's dict
merged_blocks = []
for block in blocks:
    geoid = block["geoid"]
    if geoid in census.data:
        merged_blocks.append({**block, **census.data[geoid]})

print(f"Merged blocks with data: {len(merged_blocks)}")

## 2. Basic Population Map

`create_map` returns a `MapResult` object. For PNG format the raw bytes are in `image_data`.

In [ ]:
map_result = create_map(
    data=merged_blocks,
    column="population",
    title="Population by Block Group — Portland, OR",
)

print(f"Format: {map_result.format}")
print(f"Image size: {len(map_result.image_data):,} bytes")

display(Image(data=map_result.image_data))

## 3. Map Different Columns

In [ ]:
income_map = create_map(
    data=merged_blocks,
    column="median_income",
    title="Median Household Income — Portland, OR",
)
display(Image(data=income_map.image_data))

In [ ]:
age_map = create_map(
    data=merged_blocks,
    column="median_age",
    title="Median Age — Portland, OR",
)
display(Image(data=age_map.image_data))

## 4. Custom Colormaps

Pass any matplotlib colormap name via `cmap`.

In [ ]:
viridis_map = create_map(
    data=merged_blocks,
    column="population",
    title="Population (viridis colormap)",
    cmap="viridis",
)
display(Image(data=viridis_map.image_data))

## 5. Basemap Styles

Options: `'CartoDB.Voyager'` (default), `'CartoDB.Positron'`, `'CartoDB.DarkMatter'`, or `None`.

In [ ]:
dark_map = create_map(
    data=merged_blocks,
    column="median_income",
    title="Median Income (Dark Basemap)",
    basemap="CartoDB.DarkMatter",
)
display(Image(data=dark_map.image_data))

In [ ]:
no_basemap = create_map(
    data=merged_blocks,
    column="population",
    title="Population (No Basemap)",
    basemap=None,
)
display(Image(data=no_basemap.image_data))

## 6. Overlay the Isochrone Boundary

Pass the isochrone GeoJSON Feature as `overlay_boundary` to show the travel-time boundary on the map.

In [ ]:
boundary_map = create_map(
    data=merged_blocks,
    column="population",
    title="Population with 15-min Drive Boundary",
    overlay_boundary=iso,
)
display(Image(data=boundary_map.image_data))

## 7. Overlay Point Markers

Add named point markers with `overlay_points`.

In [ ]:
points = [
    {"lat": 45.5152, "lon": -122.6784, "name": "Downtown Portland"},
    {"lat": 45.5231, "lon": -122.6765, "name": "Pearl District"},
]

points_map = create_map(
    data=merged_blocks,
    column="population",
    title="Population with Points of Interest",
    overlay_boundary=iso,
    overlay_points=points,
)
display(Image(data=points_map.image_data))

## 8. Show Statistics

Enable `show_stats=True` for an automatic stats box, or supply your own `stats_dict`.

In [ ]:
stats_map = create_map(
    data=merged_blocks,
    column="population",
    title="Population with Statistics",
    overlay_boundary=iso,
    show_stats=True,
)
display(Image(data=stats_map.image_data))

In [ ]:
# Custom statistics
import pandas as pd
df = pd.DataFrame(merged_blocks)

custom_stats = {
    "Total Population": f"{df['population'].sum():,.0f}",
    "Block Groups": str(len(df)),
    "Avg Income": f"${df['median_income'].mean():,.0f}",
}

custom_map = create_map(
    data=merged_blocks,
    column="population",
    title="Portland — Custom Stats",
    show_stats=True,
    stats_dict=custom_stats,
)
display(Image(data=custom_map.image_data))

## 9. Export to GeoJSON

In [ ]:
geojson_result = create_map(
    data=merged_blocks,
    column="population",
    export_format="geojson",
)

print(f"Format: {geojson_result.format}")
print(f"Type: {geojson_result.geojson_data['type']}")
print(f"Features: {len(geojson_result.geojson_data['features'])}")

## 10. Save to File

In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    save_path = os.path.join(tmpdir, "portland_population.png")
    saved = create_map(
        data=merged_blocks,
        column="population",
        title="Portland Population",
        save_path=save_path,
    )
    print(f"Saved to: {saved.file_path}")
    print(f"File exists: {saved.file_path.exists()}")
    print(f"File size: {saved.file_path.stat().st_size:,} bytes")

## 11. Interactive HTML Map

Set `export_format="html"` for an interactive Leaflet map. Requires `socialmapper[interactive]` (folium).

In [ ]:
try:
    html_result = create_map(
        data=merged_blocks,
        column="population",
        title="Portland Population (Interactive)",
        export_format="html",
        overlay_boundary=iso,
    )
    display(HTML(html_result.html_content))
except ImportError:
    print("Install folium for interactive maps: pip install socialmapper[interactive]")

## Summary

| What you learned | API |
|---|---|
| Create a choropleth map | `create_map(data, column)` |
| Display inline in Jupyter | `display(Image(data=result.image_data))` |
| Custom colormap & basemap | `cmap="viridis"`, `basemap="CartoDB.DarkMatter"` |
| Overlay boundary & points | `overlay_boundary=iso`, `overlay_points=[...]` |
| Statistics box | `show_stats=True`, `stats_dict={...}` |
| Export GeoJSON | `export_format="geojson"` |
| Interactive HTML | `export_format="html"` (requires folium) |

**Next notebook:** [05 — Points of Interest](05-points-of-interest.ipynb)